# 🧠 MemoryVLA Cross-Environment Generalization Test

**목표**: LIBERO에서 학습한 MemoryVLA 체크포인트를 ManiSkill2 환경에서 테스트

**핵심 포인트**:
- 🤖 **Same Robot**: Franka Panda (LIBERO = ManiSkill2)
- 🌍 **Different Environment**: LIBERO → ManiSkill2 (unseen environment)
- 📊 **Test**: 환경 일반화 능력 (Generalizability)

**Requirements**: Colab Pro with A100 GPU (80GB VRAM)

## 1️⃣ GPU 확인

In [ ]:
# GPU 확인
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2️⃣ 레포지토리 클론 (Cross-Env Generalization 브랜치)

In [ ]:
import os

# 작업 디렉토리 설정
%cd /content

# 기존 폴더가 있으면 삭제
!rm -rf MemoryVLA

# Cross-Environment Generalization 브랜치 클론
!git clone -b claude/test-libero-maniskill-generalization-cwwIv https://github.com/trillion-boy/MemoryVLA.git

%cd MemoryVLA
!git log --oneline -3

## 3️⃣ 기본 의존성 설치

In [ ]:
# PyTorch는 Colab에 이미 설치되어 있음 (CUDA 지원)
# MemoryVLA 설치
!pip install -e . -q

# 추가 의존성
!pip install transforms3d -q

## 4️⃣ ManiSkill2 설치

ManiSkill2는 Franka Panda 로봇을 기본으로 지원합니다.

In [ ]:
# ManiSkill2 설치
!pip install mani-skill2 -q

# Vulkan/렌더링 의존성 (Colab에서 필요할 수 있음)
!apt-get update -qq
!apt-get install -y -qq libegl1-mesa libgl1-mesa-dev libgles2-mesa-dev

# ManiSkill2 assets 다운로드
!python -m mani_skill2.utils.download_asset all -y

In [ ]:
# ManiSkill2 설치 확인
import gymnasium as gym
import mani_skill2.envs

# 테스트 환경 생성
env = gym.make("PickCube-v0", obs_mode="rgbd", control_mode="pd_ee_delta_pose")
obs, info = env.reset()

print("✅ ManiSkill2 설치 완료!")
print(f"Environment: PickCube-v0")
print(f"Observation keys: {obs.keys()}")
print(f"Action space: {env.action_space}")

env.close()

## 5️⃣ LIBERO-Spatial 체크포인트 다운로드

HuggingFace에서 MemoryVLA LIBERO-Spatial 체크포인트를 다운로드합니다.

In [ ]:
# HuggingFace에서 LIBERO-Spatial 체크포인트 다운로드
!pip install huggingface_hub -q

from huggingface_hub import snapshot_download
import os

# 체크포인트 저장 경로
CKPT_DIR = "/content/checkpoints/libero_spatial"
os.makedirs(CKPT_DIR, exist_ok=True)

# MemoryVLA LIBERO-Spatial 체크포인트 다운로드
# HuggingFace repo: shihao1895/memvla-libero-spatial
snapshot_download(
    repo_id="shihao1895/memvla-libero-spatial",
    local_dir=CKPT_DIR,
    local_dir_use_symlinks=False
)

print(f"\n✅ 체크포인트 다운로드 완료!")
print(f"위치: {CKPT_DIR}")
!ls -la {CKPT_DIR}

In [ ]:
# 체크포인트 파일 찾기
import glob

ckpt_files = glob.glob(f"{CKPT_DIR}/**/*.pt", recursive=True)
print("Available checkpoints:")
for f in ckpt_files:
    print(f"  - {f}")

# 가장 최근 체크포인트 선택 (또는 수동으로 지정)
if ckpt_files:
    CKPT_PATH = ckpt_files[0]  # 첫 번째 체크포인트 사용
    print(f"\n선택된 체크포인트: {CKPT_PATH}")
else:
    print("❌ 체크포인트를 찾을 수 없습니다!")

## 6️⃣ 평가 설정

In [ ]:
# ============================================
# 평가 설정 (필요시 수정)
# ============================================

# 체크포인트 경로 (위에서 자동 설정됨, 또는 수동 지정)
# CKPT_PATH = "/content/checkpoints/libero_spatial/step-XXXXX.pt"

# Unnormalization 키 (학습 데이터셋에 맞게 설정)
# LIBERO-Spatial -> libero_spatial_no_noops
# LIBERO-Object  -> libero_object_no_noops
# LIBERO-Goal    -> libero_goal_no_noops
# LIBERO-100     -> libero_90_no_noops
UNNORM_KEY = "libero_spatial_no_noops"

# 평가 에피소드 수 (빠른 테스트는 10, 정확한 평가는 50)
NUM_EPISODES = 10  # 빠른 테스트용, 실제 평가시 50으로 변경

# 결과 저장 디렉토리
EVAL_DIR = "/content/eval_results/maniskill2_generalization"
os.makedirs(EVAL_DIR, exist_ok=True)

print(f"체크포인트: {CKPT_PATH}")
print(f"Unnorm Key: {UNNORM_KEY}")
print(f"에피소드 수: {NUM_EPISODES}")
print(f"결과 저장: {EVAL_DIR}")

## 7️⃣ Cross-Environment Generalization 평가 실행

LIBERO에서 학습한 모델을 ManiSkill2 환경에서 테스트합니다.

### 테스트 환경:
| 환경 | 설명 | 난이도 |
|-----|------|-------|
| PickCube-v0 | 큐브 집기 | Easy |
| StackCube-v0 | 큐브 쌓기 | Medium |
| PickSingleYCB-v0 | YCB 물체 집기 | Medium |
| PickSingleEGAD-v0 | EGAD 물체 집기 | Hard |
| PickClutterYCB-v0 | 복잡한 장면 | Hard |

In [ ]:
# Task 1: PickCube-v0 (기본 큐브 집기)
print("="*60)
print("[1/5] PickCube-v0 평가 중...")
print("="*60)

!python evaluation/maniskill2/maniskill2_evaluator.py \
    --ckpt-path {CKPT_PATH} \
    --env-name "PickCube-v0" \
    --task-instruction "Grasp the small red cube on the table with the gripper, lift it up, and move it to the green target sphere hovering in the air." \
    --unnorm-key {UNNORM_KEY} \
    --num-episodes {NUM_EPISODES} \
    --max-steps 100 \
    --save-dir {EVAL_DIR}

In [ ]:
# Task 2: StackCube-v0 (큐브 쌓기)
print("="*60)
print("[2/5] StackCube-v0 평가 중...")
print("="*60)

!python evaluation/maniskill2/maniskill2_evaluator.py \
    --ckpt-path {CKPT_PATH} \
    --env-name "StackCube-v0" \
    --task-instruction "Pick up the small red cube from the table and carefully stack it on top of the green cube. Align the red cube above the green cube and place it down steadily." \
    --unnorm-key {UNNORM_KEY} \
    --num-episodes {NUM_EPISODES} \
    --max-steps 150 \
    --save-dir {EVAL_DIR}

In [ ]:
# Task 3: PickSingleYCB-v0 (YCB 물체 집기 - unseen objects)
print("="*60)
print("[3/5] PickSingleYCB-v0 평가 중...")
print("="*60)

!python evaluation/maniskill2/maniskill2_evaluator.py \
    --ckpt-path {CKPT_PATH} \
    --env-name "PickSingleYCB-v0" \
    --task-instruction "Grasp the object on the table with the gripper and lift it up to the goal position above the table." \
    --unnorm-key {UNNORM_KEY} \
    --num-episodes {NUM_EPISODES} \
    --max-steps 100 \
    --save-dir {EVAL_DIR}

In [ ]:
# Task 4: PickSingleEGAD-v0 (EGAD 물체 - 더 다양한 형태)
print("="*60)
print("[4/5] PickSingleEGAD-v0 평가 중...")
print("="*60)

!python evaluation/maniskill2/maniskill2_evaluator.py \
    --ckpt-path {CKPT_PATH} \
    --env-name "PickSingleEGAD-v0" \
    --task-instruction "Grasp the object on the table with the gripper and lift it up to the goal position above the table." \
    --unnorm-key {UNNORM_KEY} \
    --num-episodes {NUM_EPISODES} \
    --max-steps 100 \
    --save-dir {EVAL_DIR}

In [ ]:
# Task 5: PickClutterYCB-v0 (복잡한 장면에서 집기)
print("="*60)
print("[5/5] PickClutterYCB-v0 평가 중...")
print("="*60)

!python evaluation/maniskill2/maniskill2_evaluator.py \
    --ckpt-path {CKPT_PATH} \
    --env-name "PickClutterYCB-v0" \
    --task-instruction "Identify the target object among the clutter on the table, grasp it with the gripper, and lift it up to the goal position." \
    --unnorm-key {UNNORM_KEY} \
    --num-episodes {NUM_EPISODES} \
    --max-steps 150 \
    --save-dir {EVAL_DIR}

## 8️⃣ 결과 요약

In [ ]:
# 결과 파일 확인
print("저장된 결과 파일:")
!ls -la {EVAL_DIR}

In [ ]:
# 결과 요약 추출
!python script/eval/maniskill2_franka/extract_maniskill2_franka_results.py \
    --eval-dir {EVAL_DIR}

## 📊 결과 해석

### Cross-Environment Generalization 성공 기준:

| 성공률 | 해석 |
|--------|------|
| > 50% | 좋은 일반화 능력 |
| 30-50% | 중간 수준 |
| < 30% | 개선 필요 |

### 참고:
- LIBERO와 ManiSkill2는 **같은 로봇 (Franka Panda)** 사용
- 하지만 **환경, 물체, 시각적 특성**이 다름
- 높은 성공률 = 모델이 환경에 과적합되지 않고 **범용적** 조작 능력 학습

## 💾 결과 저장 (Google Drive로 백업)

In [ ]:
# Google Drive 마운트 (선택사항)
from google.colab import drive
drive.mount('/content/drive')

# 결과를 Google Drive로 복사
!cp -r {EVAL_DIR} /content/drive/MyDrive/memoryvla_eval_results/
print("✅ 결과가 Google Drive에 저장되었습니다!")